In [5]:
!pip install open_clip_torch -q

In [ ]:
import os
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from sklearn.metrics import accuracy_score, confusion_matrix

import open_clip
from torch.optim.swa_utils import AveragedModel, SWALR

# ==========================
# Configuration and Hyperparameters
# ==========================

START_EPOCH = 0
END_EPOCH = 0

TRAIN_FAKE_DIR = '/kaggle/input/datasets/darkwarrior10/huggingfacedeepfakefeatureextractdataset/balanced_dataset/fake'
TRAIN_REAL_DIR = '/kaggle/input/datasets/darkwarrior10/huggingfacedeepfakefeatureextractdataset/balanced_dataset/real'
VAL_FAKE_DIR = '/kaggle/input/spcup25-validation/valid/fake'
VAL_REAL_DIR = '/kaggle/input/spcup25-validation/valid/real'

# NOTE: /kaggle/input/ is read-only. Use /kaggle/working/ for saving.
BEST_MODEL_SAVE_PATH = '/kaggle/working/best_openclip_model.pth'        # Best regular model
BEST_MODEL_LOAD_PATH = '/kaggle/input/models/darkwarrior10/bestopencliponkaggledataset/pytorch/default/1/best_openclip_model.pth'
BEST_SWA_MODEL_SAVE_PATH = '/kaggle/working/best_swa_openclip_model.pth' # Best SWA model
FINAL_SWA_MODEL_SAVE_PATH = '/kaggle/working/final_swa_openclip_model.pth' # Final SWA (after BN update)

BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0
PATIENCE = 8
UNFROZEN_LAYERS = 14  # Number of transformer layers to unfreeze

SWA_START = START_EPOCH 
SWA_FREQ = 1
SWA_LR_FACTOR = 0.1

DEFAULT_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                         (0.26862954, 0.26130258, 0.27577711))
])

TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                         (0.26862954, 0.26130258, 0.27577711))
])

# ==========================
# Custom Dataset
# ==========================

class CustomDataset(Dataset):
    def __init__(self, fake_dir, real_dir, transform=None):
        self.transform = transform
        self.real_images = sorted([
            os.path.join(real_dir, img) for img in os.listdir(real_dir)
            if img.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.fake_images = sorted([
            os.path.join(fake_dir, img) for img in os.listdir(fake_dir)
            if img.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.images = self.real_images + self.fake_images
        self.labels = [1] * len(self.real_images) + [0] * len(self.fake_images)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = self.images[idx]
        label = self.labels[idx]
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            image = Image.new("RGB", (224, 224), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        return image, label

# ==========================
# Paired Sampler (Ensures Balanced Batches)
# ==========================

class PairedIndicesSampler(Sampler):
    def __init__(self, data_source, batch_size):
        self.data_source = data_source
        self.batch_size = batch_size
        self.half_length = len(data_source) // 2  # Assuming balanced classes

    def __iter__(self):
        indices_real = list(range(self.half_length))
        indices_fake = list(range(self.half_length, len(self.data_source)))
        random.shuffle(indices_real)
        random.shuffle(indices_fake)
        paired = list(zip(indices_real, indices_fake))
        flat_list = [idx for pair in paired for idx in pair]
        # If batch_size is odd, drop the last sample to maintain balance
        if len(flat_list) % self.batch_size != 0:
            flat_list = flat_list[:-(len(flat_list) % self.batch_size)]
        batches = [flat_list[i:i + self.batch_size] for i in range(0, len(flat_list), self.batch_size)]
        return iter(batches)

    def __len__(self):
        return len(self.data_source) // self.batch_size

# ==========================
# Custom Classification Model
# ==========================

class OpenCLIPClassifier(nn.Module):
    def __init__(self, openclip_model, num_classes=2):
        super(OpenCLIPClassifier, self).__init__()
        self.image_encoder = openclip_model.visual

        # Determine feature dimension
        self.image_encoder.eval()  # do not update running stats
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 224, 224)
            features = self.image_encoder(dummy_input)
            feature_dim = features.shape[-1]

        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        features = self.image_encoder(x)
        logits = self.classifier(features)
        return logits

# ==========================
# Dataloaders
# ==========================

def create_data_loaders(train_fake_dir, train_real_dir, val_fake_dir, val_real_dir,
                        train_transform, val_transform, batch_size, num_workers=4):
    train_dataset = CustomDataset(fake_dir=train_fake_dir,
                                  real_dir=train_real_dir,
                                  transform=train_transform)
    valid_dataset = CustomDataset(fake_dir=val_fake_dir,
                                  real_dir=val_real_dir,
                                  transform=val_transform)

    train_sampler = PairedIndicesSampler(train_dataset, batch_size)
    train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, 
                              num_workers=num_workers, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, 
                              num_workers=num_workers, pin_memory=True)
    return train_loader, valid_loader

# ==========================
# Evaluation + Confusion Matrix
# ==========================

def evaluate_and_plot_confusion_matrix(model, dataloader, device, class_names=['Fake', 'Real']):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc='Evaluating', leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix on Validation Set")
    plt.show()

    test_acc = accuracy_score(all_labels, all_preds)
    print(f"Validation Accuracy: {test_acc:.4f}")
    return test_acc

# ==========================
# Set Trainable Layers
# ==========================

def set_trainable_layers(model, num_layers_to_unfreeze=6):
    model_to_modify = model.module if isinstance(model, torch.nn.DataParallel) else model

    # Freeze everything
    for param in model_to_modify.image_encoder.parameters():
        param.requires_grad = False

    # Classifier always trainable
    for param in model_to_modify.classifier.parameters():
        param.requires_grad = True

    # Unfreeze from the end of the transformer
    if hasattr(model_to_modify.image_encoder, 'transformer') \
       and hasattr(model_to_modify.image_encoder.transformer, 'resblocks'):
        total_blocks = len(model_to_modify.image_encoder.transformer.resblocks)
        if num_layers_to_unfreeze > total_blocks:
            print(f"Requested to unfreeze {num_layers_to_unfreeze} layers, but only {total_blocks} available.")
            num_layers_to_unfreeze = total_blocks
        for block in model_to_modify.image_encoder.transformer.resblocks[-num_layers_to_unfreeze:]:
            for param in block.parameters():
                param.requires_grad = True
    else:
        print("Warning: Model does not have 'transformer.resblocks' - check architecture.")

# ==========================
# Count Parameters
# ==========================

def count_parameters(model):
    model_to_inspect = model.module if isinstance(model, nn.DataParallel) else model
    total_params = sum(p.numel() for p in model_to_inspect.parameters())
    trainable_params = sum(p.numel() for p in model_to_inspect.parameters() if p.requires_grad)
    percentage = (trainable_params / total_params) * 100 if total_params > 0 else 0
    return total_params, trainable_params, percentage

# ==========================
# Updated Training Function 
#   - Saves best regular model
#   - Saves best SWA model
#   - Saves final SWA model
# ==========================

def train_model(model, 
                criterion, 
                optimizer, 
                train_loader, 
                valid_loader, 
                start_epoch, 
                end_epoch, 
                patience,
                best_model_save_path,      
                final_swa_model_save_path, 
                device, 
                total_layers, 
                initial_best_val_acc=0.0,
                best_swa_model_save_path=None  
               ):
    """
    Trains the model with optional SWA. Saves:
      - Best regular model weights (best_model_save_path)
      - Best SWA model weights (best_swa_model_save_path)
      - Final SWA model weights (final_swa_model_save_path) after BN update.

    Returns:
      - final trained 'regular' model
      - final SWA model (with BN updated, if SWA started)
    """
    # If we didn't specify a separate path for best SWA model, create one
    if best_swa_model_save_path is None:
        best_swa_model_save_path = os.path.join(os.path.dirname(final_swa_model_save_path), 
                                                "best_swa_openclip_model.pth")

    best_val_acc = initial_best_val_acc       # best for regular model
    best_swa_val_acc = 0.0                   # best for SWA model
    patience_counter = 0
    train_accuracies, val_accuracies = [], []
    swa_val_accuracies = []
    swa_started = False

    scaler = torch.cuda.amp.GradScaler()

    # Prepare SWA
    swa_model = AveragedModel(model)
    swa_scheduler = None

    # Already-optimized parameters
    optimized_params = set()
    for group in optimizer.param_groups:
        for p in group['params']:
            optimized_params.add(p)

    for epoch in range(start_epoch, end_epoch + 1):
        print(f"\nEpoch {epoch}/{end_epoch}")

        # Example: dynamic unfreezing strategy
        num_layers_to_unfreeze = min(2 + epoch // 2, total_layers)
        set_trainable_layers(model, num_layers_to_unfreeze)
        print(f"Unfreezing {num_layers_to_unfreeze} layer(s) out of {total_layers}.")

        # If new layers got unfrozen, add them to the optimizer
        model_to_modify = model.module if isinstance(model, torch.nn.DataParallel) else model
        for name, param in model_to_modify.named_parameters():
            if param.requires_grad and param not in optimized_params:
                if swa_started:
                    optimizer.add_param_group({'params': param, 'swa_lr': LEARNING_RATE * SWA_LR_FACTOR})
                else:
                    optimizer.add_param_group({'params': param})
                optimized_params.add(param)
                print(f"Added '{name}' to optimizer.")

        # Training
        model.train()
        running_loss = 0.0
        train_preds, train_labels = [], []
        for inputs, labels in tqdm(train_loader, desc='Training', leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)
        train_accuracies.append(train_acc)
        avg_train_loss = running_loss / len(train_loader)
        print(f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f}")

        # Validation
        model.eval()
        val_loss = 0.0
        val_preds, val_labels = [], []
        with torch.no_grad():
            for inputs, labels in tqdm(valid_loader, desc='Validating', leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                with torch.cuda.amp.autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = outputs.argmax(dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        val_acc = accuracy_score(val_labels, val_preds)
        val_accuracies.append(val_acc)
        avg_val_loss = val_loss / len(valid_loader)
        print(f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")

        # SWA
        if epoch >= SWA_START:
            if not swa_started:
                print("Initializing SWA model and scheduler...")
                swa_model = AveragedModel(model)
                swa_scheduler = SWALR(optimizer, swa_lr=LEARNING_RATE * SWA_LR_FACTOR)
                swa_started = True
            else:
                swa_model.update_parameters(model)
                print("SWA model updated.")

            if swa_scheduler:
                swa_scheduler.step()

            # Evaluate SWA
            swa_model.eval()
            swa_val_loss = 0.0
            swa_val_preds, swa_val_labels = [], []
            with torch.no_grad():
                for inputs, labels in tqdm(valid_loader, desc='SWA Validating', leave=False):
                    inputs, labels = inputs.to(device), labels.to(device)
                    with torch.cuda.amp.autocast():
                        outputs = swa_model(inputs)
                        loss = criterion(outputs, labels)
                    swa_val_loss += loss.item()
                    preds = outputs.argmax(dim=1)
                    swa_val_preds.extend(preds.cpu().numpy())
                    swa_val_labels.extend(labels.cpu().numpy())

            swa_val_acc = accuracy_score(swa_val_labels, swa_val_preds)
            swa_val_accuracies.append(swa_val_acc)
            avg_swa_val_loss = swa_val_loss / len(valid_loader)
            print(f"SWA Val Loss: {avg_swa_val_loss:.4f}, SWA Val Acc: {swa_val_acc:.4f}")

        # Check improvement for the regular model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), best_model_save_path)
            print(f"Best regular model saved with Val Acc: {best_val_acc:.4f}")
        else:
            patience_counter += 1
            print(f"No improvement in regular model. Patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

        # Check improvement for SWA model
        if swa_started:
            if swa_val_acc > best_swa_val_acc:
                best_swa_val_acc = swa_val_acc
                torch.save(swa_model.state_dict(), best_swa_model_save_path)
                print(f"Best SWA model saved with SWA Val Acc: {swa_val_acc:.4f}")

        # Show how many parameters are trainable
        total_params, trainable_params, percentage = count_parameters(model)
        print(f"Unfrozen Parameters: {trainable_params} / {total_params} ({percentage:.2f}%)")

    # Finalize SWA: update BN stats and save final SWA
    if swa_started:
        print("Finalizing SWA model (updating BN stats)...")
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)
        torch.save(swa_model.state_dict(), final_swa_model_save_path)
        print(f"FINAL SWA model saved to {final_swa_model_save_path}")

    print("\nTraining complete.")
    print(f"Best Regular Model Val Accuracy: {best_val_acc:.4f}")
    if swa_started:
        print(f"Best SWA Model Val Accuracy: {best_swa_val_acc:.4f}")
    else:
        print("SWA did not start; SWA_START not reached.")

    # Plot Accuracy
    plt.figure(figsize=(10, 5))
    epochs_range = range(start_epoch, start_epoch + len(train_accuracies))
    plt.plot(epochs_range, train_accuracies, label='Train Accuracy')
    plt.plot(epochs_range, val_accuracies, label='Val Accuracy (Regular)')
    if swa_val_accuracies:
        plt.plot(epochs_range, swa_val_accuracies, label='Val Accuracy (SWA)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

    return model, swa_model if swa_started else None

# ==========================
# Utility: Evaluate from Checkpoint
# ==========================

def evaluate_model_from_checkpoint(model, checkpoint_path, dataloader, device, class_names=['Fake','Real']):
    """
    Loads model weights from a checkpoint, evaluates, plots confusion matrix, 
    and returns the accuracy. If checkpoint_path does not exist, returns None.
    """
    if not os.path.exists(checkpoint_path):
        print(f"No checkpoint found at {checkpoint_path}. Skipping.")
        return None
    
    # Load checkpoint
    ckpt = torch.load(checkpoint_path, map_location=device)
    # If saved as DataParallel, remove 'module.' prefix
    if any(k.startswith('module.') for k in ckpt.keys()):
        new_state_dict = {}
        for k, v in ckpt.items():
            new_key = k.replace('module.', '')
            new_state_dict[new_key] = v
        ckpt = new_state_dict
    
    model.load_state_dict(ckpt, strict=False)
    print(f"Loaded checkpoint from {checkpoint_path}")
    acc = evaluate_and_plot_confusion_matrix(model, dataloader, device, class_names)
    return acc

# ==========================
# Inference Example
# ==========================

def inference_all_checkpoints(model, 
                              swa_model, 
                              valid_loader, 
                              device,
                              best_model_save_path,
                              best_swa_model_save_path,
                              final_swa_model_save_path,
                              class_names=['Fake','Real']):
    """
    Evaluate on all known checkpoints:
      - Best Regular Model
      - Best SWA Model
      - Final SWA Model
    Returns a dict with the accuracies (or None if no checkpoint).
    """
    results = {}

    # 1) Best Regular Model
    print("\nEvaluating Best Regular Model:")
    results['best_regular'] = evaluate_model_from_checkpoint(
        model, 
        best_model_save_path, 
        valid_loader, 
        device, 
        class_names
    )

    # 2) Best SWA Model
    if swa_model is not None:
        print("\nEvaluating Best SWA Model:")
        results['best_swa'] = evaluate_model_from_checkpoint(
            swa_model, 
            best_swa_model_save_path, 
            valid_loader, 
            device, 
            class_names
        )
    else:
        print("\nNo SWA model available to evaluate for best SWA checkpoint.")
        results['best_swa'] = None

    # 3) Final SWA Model
    if swa_model is not None:
        print("\nEvaluating Final SWA Model (with BN updated):")
        results['final_swa'] = evaluate_model_from_checkpoint(
            swa_model, 
            final_swa_model_save_path, 
            valid_loader, 
            device, 
            class_names
        )
    else:
        print("\nNo SWA model available to evaluate for final SWA checkpoint.")
        results['final_swa'] = None

    return results

# ==========================
# Main Execution
# ==========================

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs available: {num_gpus}")

    effective_batch_size = BATCH_SIZE
    if num_gpus > 1:
        if BATCH_SIZE % num_gpus != 0:
            effective_batch_size = BATCH_SIZE - (BATCH_SIZE % num_gpus)
            print(f"Adjusted batch size to {effective_batch_size} for {num_gpus} GPUs.")
    else:
        effective_batch_size = BATCH_SIZE

    train_loader, valid_loader = create_data_loaders(
        TRAIN_FAKE_DIR,
        TRAIN_REAL_DIR,
        VAL_FAKE_DIR,
        VAL_REAL_DIR,
        TRAIN_TRANSFORM,
        DEFAULT_TRANSFORM,
        batch_size=effective_batch_size,
        num_workers=4
    )

    # Create OpenCLIP model
    openclip_model, _, _ = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
    classifier_model = OpenCLIPClassifier(openclip_model, num_classes=2).to(device)

    # DataParallel if multiple GPUs
    if num_gpus > 1:
        print(f"Using DataParallel over {num_gpus} GPUs!")
        classifier_model = nn.DataParallel(classifier_model)

    # Attempt to load pre-trained weights
    if os.path.exists(BEST_MODEL_LOAD_PATH):
        try:
            state_dict = torch.load(BEST_MODEL_LOAD_PATH, map_location=device)
            if isinstance(state_dict, dict) and any(k.startswith('module.') for k in state_dict.keys()):
                new_state_dict = {}
                for k, v in state_dict.items():
                    new_key = k.replace('module.', '')
                    new_state_dict[new_key] = v
                state_dict = new_state_dict
            classifier_model.load_state_dict(state_dict, strict=False)
            print(f"Loaded pre-trained model from {BEST_MODEL_LOAD_PATH}")
        except RuntimeError as e:
            print(f"Error loading {BEST_MODEL_LOAD_PATH}: {e}")
    else:
        print(f"No pre-trained model found at {BEST_MODEL_LOAD_PATH}; starting from scratch.")

    # Quick initial validation (optional)
    print("\nInitial Validation (Regular Model) - if weights loaded:")
    initial_val_acc = evaluate_and_plot_confusion_matrix(classifier_model, valid_loader, device)

    set_trainable_layers(classifier_model, num_layers_to_unfreeze=2)
    total_params, trainable_params, percentage = count_parameters(classifier_model)
    print(f"After initial unfreeze, trainable params: {trainable_params:,}/{total_params:,} ({percentage:.2f}%)")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, classifier_model.parameters()),
                            lr=LEARNING_RATE,
                            weight_decay=WEIGHT_DECAY)

    # Train
    trained_model, final_swa_model = train_model(
        model=classifier_model,
        criterion=criterion,
        optimizer=optimizer,
        train_loader=train_loader,
        valid_loader=valid_loader,
        start_epoch=START_EPOCH,
        end_epoch=END_EPOCH,
        patience=PATIENCE,
        best_model_save_path=BEST_MODEL_SAVE_PATH,
        final_swa_model_save_path=FINAL_SWA_MODEL_SAVE_PATH,
        device=device,
        total_layers=UNFROZEN_LAYERS,
        initial_best_val_acc=initial_val_acc,
        best_swa_model_save_path=BEST_SWA_MODEL_SAVE_PATH
    )

    # Post-Training: Evaluate ALL saved checkpoints
    print("\nEvaluating all saved checkpoints:")
    results = inference_all_checkpoints(
        model=trained_model,
        swa_model=final_swa_model,
        valid_loader=valid_loader,
        device=device,
        best_model_save_path=BEST_MODEL_SAVE_PATH,
        best_swa_model_save_path=BEST_SWA_MODEL_SAVE_PATH,
        final_swa_model_save_path=FINAL_SWA_MODEL_SAVE_PATH,
        class_names=['Fake','Real']
    )
    print(f"\nResults: {results}")

    print("\nDone.")

if __name__ == "__main__":
    main()


In [ ]:
import os
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from sklearn.metrics import accuracy_score, confusion_matrix

import open_clip

# ==========================
# Configuration and Hyperparameters
# ==========================

TRAIN_FAKE_DIR = '/kaggle/input/datasets/darkwarrior10/huggingfacedeepfakefeatureextractdataset/balanced_dataset/fake'
TRAIN_REAL_DIR = '/kaggle/input/datasets/darkwarrior10/huggingfacedeepfakefeatureextractdataset/balanced_dataset/real'
VAL_FAKE_DIR = '/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Validation/Fake'
VAL_REAL_DIR = '/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Validation/Real'

# Load the trained model from this path
BEST_MODEL_LOAD_PATH = '/kaggle/input/models/darkwarrior10/bestopencliponkaggledataset/pytorch/default/5/best_openclip_model.pth'

BATCH_SIZE = 32

DEFAULT_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                         (0.26862954, 0.26130258, 0.27577711))
])

# ==========================
# Custom Dataset
# ==========================

class CustomDataset(Dataset):
    def __init__(self, fake_dir, real_dir, transform=None):
        self.transform = transform
        self.real_images = sorted([
            os.path.join(real_dir, img) for img in os.listdir(real_dir)
            if img.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.fake_images = sorted([
            os.path.join(fake_dir, img) for img in os.listdir(fake_dir)
            if img.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.images = self.real_images + self.fake_images
        self.labels = [1] * len(self.real_images) + [0] * len(self.fake_images)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = self.images[idx]
        label = self.labels[idx]
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            image = Image.new("RGB", (224, 224), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        return image, label

# ==========================
# Custom Classification Model
# ==========================

class OpenCLIPClassifier(nn.Module):
    def __init__(self, openclip_model, num_classes=2):
        super(OpenCLIPClassifier, self).__init__()
        self.image_encoder = openclip_model.visual

        # Determine feature dimension
        self.image_encoder.eval()
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 224, 224)
            features = self.image_encoder(dummy_input)
            feature_dim = features.shape[-1]

        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        features = self.image_encoder(x)
        logits = self.classifier(features)
        return logits

# ==========================
# Dataloaders
# ==========================

def create_data_loaders(val_fake_dir, val_real_dir, val_transform, batch_size, num_workers=4):
    valid_dataset = CustomDataset(fake_dir=val_fake_dir,
                                  real_dir=val_real_dir,
                                  transform=val_transform)

    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, 
                              num_workers=num_workers, pin_memory=True)
    return valid_loader

# ==========================
# Evaluation + Confusion Matrix
# ==========================

def evaluate_and_plot_confusion_matrix(model, dataloader, device, class_names=['Fake', 'Real']):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc='Evaluating', leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix on Validation Set")
    plt.show()

    test_acc = accuracy_score(all_labels, all_preds)
    print(f"Validation Accuracy: {test_acc:.4f}")
    return test_acc

# ==========================
# Load Model from Checkpoint
# ==========================

def load_model_from_checkpoint(model, checkpoint_path, device):
    """
    Loads model weights from a checkpoint.
    """
    if not os.path.exists(checkpoint_path):
        print(f"No checkpoint found at {checkpoint_path}.")
        return None
    
    # Load checkpoint
    ckpt = torch.load(checkpoint_path, map_location=device)
    
    # If saved as DataParallel, remove 'module.' prefix
    if any(k.startswith('module.') for k in ckpt.keys()):
        new_state_dict = {}
        for k, v in ckpt.items():
            new_key = k.replace('module.', '')
            new_state_dict[new_key] = v
        ckpt = new_state_dict
    
    model.load_state_dict(ckpt, strict=False)
    print(f"Loaded checkpoint from {checkpoint_path}")
    return model

# ==========================
# Main Execution - Validation Only
# ==========================

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Create validation dataloader
    valid_loader = create_data_loaders(
        VAL_FAKE_DIR,
        VAL_REAL_DIR,
        DEFAULT_TRANSFORM,
        batch_size=BATCH_SIZE,
        num_workers=4
    )
    
    print(f"Validation dataset size: {len(valid_loader.dataset)}")

    # Create OpenCLIP model
    print("Creating OpenCLIP model...")
    openclip_model, _, _ = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
    classifier_model = OpenCLIPClassifier(openclip_model, num_classes=2).to(device)

    # Load the trained model weights
    print("Loading trained model weights...")
    classifier_model = load_model_from_checkpoint(
        classifier_model, 
        BEST_MODEL_LOAD_PATH, 
        device
    )
    
    if classifier_model is None:
        print("Failed to load model. Exiting.")
        return

    # Evaluate the model
    print("\nEvaluating loaded model:")
    val_acc = evaluate_and_plot_confusion_matrix(
        classifier_model, 
        valid_loader, 
        device,
        class_names=['Fake', 'Real']
    )
    
    print(f"\nFinal Validation Accuracy: {val_acc:.4f}")
    print("\nDone.")

if __name__ == "__main__":
    main()

Using device: cuda
Validation dataset size: 39428
Creating OpenCLIP model...


/opt/conda/lib/python3.10/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Loading trained model weights...


/tmp/ipykernel_34/1797920149.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location=device)


Loaded checkpoint from /kaggle/input/models/darkwarrior10/bestopencliponkaggledataset/pytorch/default/5/best_openclip_model.pth

Evaluating loaded model:


Evaluating:  95%|█████████▍| 1167/1233 [27:37<01:33,  1.42s/it]